In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.days_until_next_meeting AS (
-- REBUILT Nov 1 2026: the CRM (lead_activites) never carries future-dated meetings
-- (its 147 'upcoming' rows all have past starts_at), so this table was permanently empty.
-- The only source of future meetings is the Calendly export. Clients are matched through
-- silver.user_identity_map (the 2_8 identity spine): invitee 32-hex hash OR real email
-- matches the spine's identity keys, and lead attribution merges activity-logged and
-- contact-record-owned leads. Pre-flight validated: 464 clients vs 434 via inline bridges.
--
-- SPEC 5.1 NOTE (Nov 6 2026, user-approved deviation): the spec constant
-- "Upcoming-meeting type = Meeting" refers to CRM activity type. The CRM never
-- carries future-dated meetings (verified across all activities — its only
-- 'upcoming'-typed rows have past start times), so that constant can never match
-- a future event and is intentionally NOT applied here. Calendly is the only
-- usable source of upcoming meetings. Deliberate design — do not "fix" by
-- switching back to a CRM type filter.
WITH coaching_clients AS (
    SELECT id AS lead_id
    FROM crm_ingestion.silver.leads_processed
    WHERE status_label = 'Coaching Client'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY id ORDER BY date_updated DESC) = 1
),
calendly_future AS (
    SELECT
        INVITEE_NAME,
        INVITEE_EMAIL,
        EVENT_START_TIME
    FROM crm_ingestion.silver.calendly_scheduled_events_processed
    WHERE EVENT_START_TIME IS NOT NULL
      AND DATE(EVENT_START_TIME) > CURRENT_TIMESTAMP
    -- Spec dedup for duplicate EVENT_URI: INVITEE_UPDATED_AT, then EVENT_CREATED_AT, then insert date
    QUALIFY ROW_NUMBER() OVER (PARTITION BY EVENT_URI ORDER BY INVITEE_UPDATED_AT DESC, EVENT_CREATED_AT DESC, bronze_insert_date DESC) = 1
),
calendly_matched AS (
    SELECT lead_id, MIN(EVENT_START_TIME) AS next_start
    FROM (
        SELECT DISTINCT
            explode(sp.lead_ids) AS lead_id,
            cf.EVENT_START_TIME
        FROM calendly_future cf
        JOIN crm_ingestion.silver.user_identity_map sp
          ON (cf.INVITEE_NAME IS NOT NULL AND array_contains(sp.user_32digits, LOWER(cf.INVITEE_NAME)))
          OR (cf.INVITEE_EMAIL IS NOT NULL AND array_contains(sp.real_emails, LOWER(cf.INVITEE_EMAIL)))
    )
    GROUP BY lead_id
)
SELECT
    cc.lead_id,
    DATEDIFF(DATE(cm.next_start), CURRENT_DATE) AS days_until_next_meeting
FROM coaching_clients cc
JOIN calendly_matched cm ON cc.lead_id = cm.lead_id
)

# Verification

Checks that the rebuilt table is populated and the values are sane.

In [0]:
display(spark.sql("""
SELECT COUNT(*) AS clients_with_next_meeting,
       COUNT(DISTINCT lead_id) AS distinct_clients,
       MIN(days_until_next_meeting) AS soonest_days,
       MAX(days_until_next_meeting) AS farthest_days,
       ROUND(AVG(days_until_next_meeting), 1) AS avg_days
FROM crm_ingestion.gold.days_until_next_meeting
"""))

display(spark.sql("""
SELECT CASE WHEN days_until_next_meeting <= 7 THEN 'within a week'
            WHEN days_until_next_meeting <= 30 THEN 'within a month'
            WHEN days_until_next_meeting <= 90 THEN 'within a quarter'
            ELSE 'beyond 90 days' END AS booking_window,
       COUNT(*) AS clients
FROM crm_ingestion.gold.days_until_next_meeting
GROUP BY 1 ORDER BY 1
"""))